# AQAL Vertex-Level Learned Transform

**Advanced notebook.** Extends `aqal_finetune_colab.ipynb` to produce a **vertex-level** learned transform (20,484 cortical points), compatible with the existing production API.

**Approach:**
1. Load v5 statistical transform (per-vertex scale + shift from t-tests)
2. Load connectivity features (4,950-dim)
3. Train a **learned residual correction** on top of v5: `nd_pred = nt_pred * (scale_v5 + delta_scale(features)) + (shift_v5 + delta_shift(features))`
4. Delta networks are small MLPs conditioned on subject features
5. Export as drop-in replacement for v5

**Why this approach:** The statistical v5 works. Rather than throw it away, we learn a correction on top that adapts per-subject. Lower risk, faster training, better interpretability.

**Runtime:** ~45 min on T4.

In [ ]:
!pip install -q huggingface_hub nilearn
import torch, numpy as np
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1. Load Data & v5 Baseline

In [ ]:
from huggingface_hub import hf_hub_download

# Data
data_path = hf_hub_download('Ibrahim9989/neurobrain-nd-transform', 'colab_training_data.npz')
data = np.load(data_path, allow_pickle=True)
X = data['X']; y = data['y']
age = data['age']; sex = data['sex']; site_idx = data['site_idx']

# v5 statistical baseline
v5_path = hf_hub_download('Ibrahim9989/neurobrain-nd-transform', 'neurodiverse_transform_v5.pt')
v5 = torch.load(v5_path, map_location='cpu', weights_only=False)

v5_scale = v5['vertex_scale'].numpy()  # (20484,)
v5_shift = v5['vertex_shift'].numpy()  # (20484,)
print(f'v5 scale: range [{v5_scale.min():.3f}, {v5_scale.max():.3f}]')
print(f'v5 shift: range [{v5_shift.min():.4f}, {v5_shift.max():.4f}]')
print(f'Data: {X.shape[0]} subjects, {X.shape[1]} features')

## 2. Build Connectivity → Vertex Mapping

The Schaefer 100-parcel atlas projects to the fsaverage5 surface (20,484 vertices). Each vertex belongs to one ROI. We reuse the mapping from v5's training.

In [ ]:
from nilearn.surface import vol_to_surf
from nilearn import datasets as ni_datasets

atlas = ni_datasets.fetch_atlas_schaefer_2018(n_rois=100)
fsaverage = ni_datasets.fetch_surf_fsaverage('fsaverage5')
left_proj = vol_to_surf(atlas.maps, fsaverage['pial_left'])
right_proj = vol_to_surf(atlas.maps, fsaverage['pial_right'])
surface_labels = np.concatenate([left_proj, right_proj]).astype(int)
print(f'Surface labels: {surface_labels.shape}, unique ROIs: {len(np.unique(surface_labels))}')

## 3. Learned Residual Correction

Per-subject deltas that refine v5's global scale/shift. Trained to minimize connectivity divergence from empirical ASD mean when applied to TD subjects.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class ResidualCorrection(nn.Module):
    """Per-ROI learned delta on top of v5 statistical transform."""
    def __init__(self, n_rois=100, cond_dim=3, hidden=256):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(cond_dim, hidden),
            nn.LayerNorm(hidden), nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(hidden, hidden),
            nn.LayerNorm(hidden), nn.GELU(),
        )
        self.scale_head = nn.Linear(hidden, n_rois)
        self.shift_head = nn.Linear(hidden, n_rois)
        # Small init so initial output is ~v5 baseline
        nn.init.zeros_(self.scale_head.weight); nn.init.zeros_(self.scale_head.bias)
        nn.init.zeros_(self.shift_head.weight); nn.init.zeros_(self.shift_head.bias)

    def forward(self, cond):
        h = self.encoder(cond)
        return self.scale_head(h) * 0.05, self.shift_head(h) * 0.01  # small corrections

model = ResidualCorrection().to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

## 4. Training

Train residual corrections so that TD subjects' connectivity, after transform, matches the ASD mean connectivity (distribution matching).

In [ ]:
# Build per-ROI aggregation: each 4,950 feature is an (i,j) pair of ROIs
pair_to_roi = []
idx = 0
for i in range(100):
    for j in range(i+1, 100):
        pair_to_roi.append((i, j))
pair_to_roi = np.array(pair_to_roi)  # (4950, 2)

# ASD vs TD mean connectivity (the target)
asd_mean = X[y == 1].mean(axis=0)  # (4950,)
td_mean = X[y == 0].mean(axis=0)   # (4950,)
target_delta = (asd_mean - td_mean)  # (4950,)

# Context features
def build_cond(age_arr, sex_arr):
    return np.stack([
        (age_arr - 18) / 20,
        (sex_arr == 1).astype(np.float32),
        (sex_arr == 2).astype(np.float32),
    ], axis=-1).astype(np.float32)

cond = build_cond(age, sex)
print(f'Conditioning: {cond.shape}')

# Split
from sklearn.model_selection import train_test_split
idx_all = np.arange(len(X))
train_idx, val_idx = train_test_split(idx_all, test_size=0.2, stratify=y, random_state=42)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

epochs = 60
lr = 1e-3
batch_size = 64

# Use only TD subjects for training (we want to learn 'how to nudge a TD brain toward ASD')
td_train_idx = train_idx[y[train_idx] == 0]
td_val_idx = val_idx[y[val_idx] == 0]

td_X = torch.FloatTensor(X[td_train_idx])
td_cond = torch.FloatTensor(cond[td_train_idx])
target = torch.FloatTensor(asd_mean).to(device)
pair_idx_t = torch.LongTensor(pair_to_roi).to(device)  # (4950, 2)

ds = TensorDataset(td_X, td_cond)
loader = DataLoader(ds, batch_size=batch_size, shuffle=True)

opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

def apply_correction(x, delta_scale, delta_shift):
    """Apply per-ROI scale+shift to pair connectivity (4950,).
    For ROI pair (i,j), use average of deltas from i and j."""
    # delta_scale, delta_shift: (batch, 100)
    s_i = delta_scale[:, pair_idx_t[:, 0]]  # (batch, 4950)
    s_j = delta_scale[:, pair_idx_t[:, 1]]
    avg_scale = 1.0 + (s_i + s_j) / 2
    b_i = delta_shift[:, pair_idx_t[:, 0]]
    b_j = delta_shift[:, pair_idx_t[:, 1]]
    avg_shift = (b_i + b_j) / 2
    return x * avg_scale + avg_shift

for epoch in range(epochs):
    model.train()
    total = 0
    for xb, cb in loader:
        xb, cb = xb.to(device), cb.to(device)
        opt.zero_grad()
        ds_, sh = model(cb)
        corrected = apply_correction(xb, ds_, sh)
        # Loss: corrected TD should look like ASD mean
        loss_mean = F.mse_loss(corrected.mean(dim=0), target)
        # Regularize: don't deviate too much from input
        loss_reg = F.mse_loss(corrected, xb) * 0.05
        loss = loss_mean + loss_reg
        loss.backward()
        opt.step()
        total += loss.item()
    scheduler.step()
    if (epoch + 1) % 10 == 0:
        print(f'  Epoch {epoch+1}/{epochs}: loss={total/len(loader):.6f}')

print('\nTraining complete.')

## 5. Convert to Vertex-Level Transform

In [ ]:
# For deployment, compute a per-age-band transform (since we don't know individual
# age at inference time in the general case — but the API can support it).
# Compute mean deltas for each age band.

model.eval()
bands = {'child': (0, 12), 'adolescent': (12, 18), 'adult': (18, 100)}
band_transforms = {}

for band_name, (lo, hi) in bands.items():
    band_mask = (age >= lo) & (age < hi)
    if band_mask.sum() < 20:
        continue
    band_cond = cond[band_mask]
    with torch.no_grad():
        c_t = torch.FloatTensor(band_cond).to(device)
        ds_, sh = model(c_t)
        roi_scale_delta = ds_.mean(dim=0).cpu().numpy()  # (100,) mean ROI scale correction
        roi_shift_delta = sh.mean(dim=0).cpu().numpy()  # (100,)

    # Expand to 20,484 vertices
    vertex_scale_learned = v5_scale.copy()
    vertex_shift_learned = v5_shift.copy()
    for roi_idx in range(100):
        verts = np.where(surface_labels == roi_idx + 1)[0]
        if len(verts) > 0:
            vertex_scale_learned[verts] += roi_scale_delta[roi_idx]
            vertex_shift_learned[verts] += roi_shift_delta[roi_idx]

    band_transforms[band_name] = {
        'vertex_scale': torch.FloatTensor(vertex_scale_learned),
        'vertex_shift': torch.FloatTensor(vertex_shift_learned),
        'n_subjects': int(band_mask.sum()),
        'roi_scale_delta': torch.FloatTensor(roi_scale_delta),
        'roi_shift_delta': torch.FloatTensor(roi_shift_delta),
    }
    print(f'{band_name}: {int(band_mask.sum())} subjects, scale range [{vertex_scale_learned.min():.3f}, {vertex_scale_learned.max():.3f}]')

# Overall (all-ages) transform
with torch.no_grad():
    all_cond = torch.FloatTensor(cond).to(device)
    ds_, sh = model(all_cond)
    roi_scale_all = ds_.mean(dim=0).cpu().numpy()
    roi_shift_all = sh.mean(dim=0).cpu().numpy()

vertex_scale_final = v5_scale.copy()
vertex_shift_final = v5_shift.copy()
for roi_idx in range(100):
    verts = np.where(surface_labels == roi_idx + 1)[0]
    if len(verts) > 0:
        vertex_scale_final[verts] += roi_scale_all[roi_idx]
        vertex_shift_final[verts] += roi_shift_all[roi_idx]

## 6. Save & Upload

In [ ]:
from datetime import datetime

checkpoint = {
    # Compatible with v5 API format — drop-in replacement
    'vertex_scale': torch.FloatTensor(vertex_scale_final),
    'vertex_shift': torch.FloatTensor(vertex_shift_final),
    'roi_effect': v5['roi_effect'],  # reuse for sensory profile
    'vertex_ci_lower': v5['vertex_ci_lower'],  # reuse uncertainty
    'vertex_ci_upper': v5['vertex_ci_upper'],
    't_stats': v5['t_stats'],
    'p_values_fdr': v5['p_values_fdr'],
    'n_asd': int(y.sum()),
    'n_td': int((1-y).sum()),
    'sig_uncorrected': int(v5.get('sig_uncorrected', 0)),
    'sig_fdr': int(v5.get('sig_fdr', 0)),
    'sig_bonferroni': int(v5.get('sig_bonferroni', 0)),
    'roi_labels': v5['roi_labels'],
    'age_transforms': band_transforms,
    # Learned model weights (for reference + retraining)
    'learned_model_state': {k: v.cpu() for k, v in model.state_dict().items()},
    'version': 'v6_learned_vertex',
    'corrections': ['site_harmonization', 'age_sex_covariates', 'fdr_bh', 'bootstrap_ci', 'learned_residual'],
    'trained_at': datetime.utcnow().isoformat(),
    'notes': 'v6: v5 statistical baseline + learned per-ROI residual correction. Trained on 1,545 subjects across 36 sites.',
}

torch.save(checkpoint, 'neurodiverse_transform_v6.pt')
import os
print(f'Saved: {os.path.getsize("neurodiverse_transform_v6.pt") / 1024:.0f} KB')

In [ ]:
from huggingface_hub import HfApi
import os

HF_TOKEN = os.environ.get('HF_TOKEN') or ''
if HF_TOKEN:
    api = HfApi(token=HF_TOKEN)
    api.upload_file(
        path_or_fileobj='neurodiverse_transform_v6.pt',
        path_in_repo='neurodiverse_transform_v6.pt',
        repo_id='Ibrahim9989/neurobrain-nd-transform',
    )
    print('Uploaded!')
else:
    print('Set HF_TOKEN env var to upload. File saved locally.')